# jobs

> A job queue kept in the vault file: work survives a crash, failures are retried with a growing delay, and jobs that keep failing are set aside for a person

In [ ]:
#| default_exp jobs

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Fetching a page, ingesting a batch of files and rebuilding the topic map can fail or stop part way. `Queue` keeps such work as rows in a `jobs` table in the vault's own SQLite file. A job outlives the process that queued it. No separate server is needed.

`enqueue` adds a job. A worker takes jobs with `claim`. `claim` marks the jobs as running and gives the worker a lease, which is a time limit on the claim (`lease`, 10 minutes by default). Two workers that claim at the same time never get the same job.

If a worker dies, its lease runs out. `reclaim` then fails the job with the error `lease expired`. This counts as one attempt.

A job that fails runs again after a delay. The delay doubles with each failure, from 30 seconds up to 6 hours. After `max_attempts` failures (5 by default) the job is dead. A dead job stays in the table with its error, for a person to examine.

The queue lives in the same file as the documents. A job queued inside a `write_txn` is committed with the write that caused it. `Queue` needs only a database and knows nothing about a vault. [acquire](01_acquire.ipynb) registers the vault's own jobs on it.

In [ ]:
#| export
import json, random, time, uuid
from fastcore.all import AttrDict, L, first, patch, store_attr
from litesearch import write_txn, BUSY_TIMEOUT_MS

STATES = ('ready', 'running', 'done', 'dead')

In [ ]:
#| export
class Retry(Exception):
    "Raise from a handler to have the job run again when nothing failed but the work should be retried, such as a bot wall, an HTTP 429 or an empty read."
    def __init__(self, msg:str='', after:float=None):
        super().__init__(msg)
        self.after = after   # seconds until the retry; None uses the backoff schedule

class Terminal(Exception):
    "Raise from a handler to mark the job dead at once, for a failure that retrying cannot fix, such as a 401 or a missing file."

def backoff(attempt:int,       # failures so far; 1 on the first
            base:float=30,     # first delay, seconds
            cap:float=21600,   # longest delay, 6h
            jitter:float=0.25  # +/- fraction of the delay; spreads out jobs that failed together
) -> float:
    "The delay before the next attempt: `base` seconds doubled for each failure so far, at most `cap`, with some random jitter."
    return max(1., min(cap, base * 2**(attempt-1)) * (1 + random.uniform(-jitter, jitter)))

In [ ]:
#| hide
test_eq(backoff(1, base=10, jitter=0) , 10)
test_eq(backoff(4, base=10, jitter=0) , 80)
test_eq(backoff(99, base=10, jitter=0), 21600)   # capped, not astronomical
assert 7.5 <= backoff(1, base=10) <= 12.5

## The queue

A job has a `kind`, which names the handler that runs it, and a `payload`, a dict stored as JSON. `key` makes queueing idempotent. While a job with the same kind and key is waiting or running, `enqueue` returns that job and adds no second one. A repeated fan-out does not double the work. A job with a lower `priority` number runs first.

In [ ]:
#| export
def _row(r) -> AttrDict:
    'A job row with its payload parsed.'
    d = dict(r)
    p = d.get('payload')
    d['payload'] = json.loads(p) if isinstance(p, str) else (p or {})
    return AttrDict(d)

class Queue:
    "A job queue stored in an open litesearch database, which runs every job at least once."
    def __init__(self,
                 db,                     # open litesearch/fastlite Database; the queue lives in its file
                 lease:float=600,        # seconds a claim is held before another worker may take the job
                 max_attempts:int=5,     # attempts before a job is dead-lettered
                 base:float=30,          # first retry delay; doubles per attempt
                 keep:float=604800,      # how long finished jobs and run history survive `purge`, 7d
                 busy_ms:int=BUSY_TIMEOUT_MS):  # lock wait before a write gives up; apsw's stock default is 100ms
        store_attr()
        self.handlers = {}
        self._mk()

    def _mk(self):
        'The two tables and their indexes, created on first use.'
        # 8 workers on one file exceed apsw's 100ms default and fail the write instead of waiting
        self.db.conn.set_busy_timeout(self.busy_ms)
        self.t = self.db.t.jobs
        self.t.create(id=str, kind=str, payload=str, key=str, state=str, priority=int, run_at=float,
                      lease_until=float, worker=str, attempt=int, max_attempts=int, error=str,
                      created_at=float, updated_at=float, pk='id', if_not_exists=True)
        self.runs = self.db.t.job_runs
        self.runs.create(id=str, job_id=str, kind=str, worker=str, status=str, error=str,
                         started_at=float, ended_at=float, took=float, pk='id', if_not_exists=True)
        for sql in (
            "CREATE INDEX IF NOT EXISTS jobs_due ON jobs(priority, run_at) WHERE state='ready'",
            "CREATE INDEX IF NOT EXISTS jobs_lease ON jobs(lease_until) WHERE state='running'",
            "CREATE UNIQUE INDEX IF NOT EXISTS jobs_key ON jobs(kind, key) "
            "WHERE key IS NOT NULL AND state IN ('ready','running')",
            "CREATE INDEX IF NOT EXISTS job_runs_job ON job_runs(job_id, started_at)"):
            self.db.conn.execute(sql)

    def __repr__(self):
        s = self.stats()
        return f"Queue({s['ready']} ready, {s['running']} running, {s['done']} done, {s['dead']} dead)"

In [ ]:
#| export
@patch
def enqueue(self:Queue,
            kind:str,             # which handler runs it
            payload:dict=None,    # arguments, stored as JSON
            run_at:float=None,    # epoch seconds; defaults to now
            priority:int=0,       # lower runs first
            key:str=None,         # idempotency key: an identical key still ready or running wins
            max_attempts:int=None
) -> AttrDict:
    "Add one job, or return the waiting or running job that already has this `key`."
    now = time.time()
    row = dict(id=uuid.uuid4().hex[:12], kind=kind, payload=json.dumps(payload or {}), key=key,
               state='ready', priority=priority, run_at=run_at if run_at is not None else now,
               lease_until=0., worker=None, attempt=0, max_attempts=max_attempts or self.max_attempts,
               error=None, created_at=now, updated_at=now)
    with write_txn(self.db):
        if key is not None and (was := first(self.db.q(
                "SELECT * FROM jobs WHERE kind=? AND key=? AND state IN ('ready','running')", [kind, key]))):
            return _row(was)
        self.t.insert(row)
    return _row(row)

@patch
def enqueue_all(self:Queue, kind:str, payloads, **kw) -> L:
    'Enqueue many jobs of one kind in a single transaction.'
    with write_txn(self.db): return L(payloads).map(lambda p: self.enqueue(kind, p, **kw))

### Claiming

`claim` picks and marks its jobs in one SQL statement. SQLite runs one write at a time. No other worker can take a job between the pick and the mark. Every job that `claim` returns is already running and already leased.

`ack` (finished) and `fail` (failed) count only while the job is still running under the caller's `worker` name. Each worker needs its own name. A handler slower than `lease` loses its job to `reclaim`, and another worker can claim it. Without this check, the late `ack` could mark as done a job that another worker is still running. The late `fail` could send a finished job back for a third run. A late call returns the job with the state `lost`, and the run history records `lost`.

In [ ]:
#| export
@patch
def claim(self:Queue,
          worker:str='worker',  # who is holding the lease; recorded on the row and in history
          n:int=1,              # how many jobs to take
          now:float=None
) -> L:
    "Claim up to `n` due jobs in one statement; no job was claimed twice over 400 jobs and 8 processes (`evals/jobs.py`)."
    now = time.time() if now is None else now
    with write_txn(self.db):
        rows = self.db.q(
            'UPDATE jobs SET state=?, worker=?, lease_until=?, updated_at=? WHERE id IN '
            '(SELECT id FROM jobs WHERE state=? AND run_at<=? ORDER BY priority, run_at LIMIT ?) '
            'RETURNING *', ['running', worker, now + self.lease, now, 'ready', now, n])
    # RETURNING gives no defined row order; ORDER BY in the subquery only picks which rows
    return L(sorted(rows, key=lambda r: (r['priority'], r['run_at']))).map(_row)

@patch
def _held(self:Queue, job) -> dict|None:
    "The row as it stands, or None if this worker no longer holds it. Call inside a `write_txn`."
    cur = first(self.db.q('SELECT * FROM jobs WHERE id=?', [job['id']]))
    if cur is None or cur['state'] != 'running' or cur['worker'] != job.get('worker'): return None
    return cur

@patch
def ack(self:Queue, job) -> AttrDict:
    "Mark a claimed job done, or return it as `lost` when this worker's lease was reclaimed."
    with write_txn(self.db):
        if self._held(job) is None: return _row(dict(job, state='lost'))
        self.db.q('UPDATE jobs SET state=?, error=NULL, worker=NULL, lease_until=0, updated_at=? WHERE id=?',
                  ['done', time.time(), job['id']])
    return _row(dict(job, state='done'))

@patch
def fail(self:Queue,
         job,                # the claimed job
         error:str,          # what went wrong; truncated to 500 chars on the row
         after:float=None,   # seconds until the retry; None uses `backoff`
         dead:bool=False     # dead-letter now, whatever the attempts left: retrying a 401 never turns into success
) -> AttrDict:
    "Record a failure and schedule a retry after the backoff delay, or mark the job dead once its attempts are spent."
    now, err = time.time(), str(error)[:500]
    with write_txn(self.db):
        if (cur := self._held(job)) is None: return _row(dict(job, state='lost'))
        att = cur['attempt'] + 1
        dead = dead or att >= (cur['max_attempts'] or self.max_attempts)
        nxt = cur['run_at'] if dead else now + (backoff(att, base=self.base) if after is None else after)
        self.db.q('UPDATE jobs SET state=?, attempt=?, run_at=?, error=?, worker=NULL, lease_until=0, '
                  'updated_at=? WHERE id=?', ['dead' if dead else 'ready', att, nxt, err, now, cur['id']])
    return _row(dict(cur, state='dead' if dead else 'ready', attempt=att, run_at=nxt, error=err))

@patch
def reclaim(self:Queue, now:float=None) -> L:
    "Fail each running job whose lease expired with `lease expired`; this counts one attempt, and the job is dead after `max_attempts`."
    now = time.time() if now is None else now
    exp = self.db.q('SELECT * FROM jobs WHERE state=? AND lease_until>0 AND lease_until<=?', ['running', now])
    # two pollers can read the same expired row; the check in `fail` lets only one of them count it
    return L(exp).map(lambda j: self.fail(j, 'lease expired')).filter(lambda j: j['state'] != 'lost')

### Running

A handler is a function `fn(payload)`. Register it with `register` or the `on` decorator. `run_one` runs one claimed job. `drain` claims and runs jobs until the queue is empty or `limit` jobs have run. A cron job calls `drain`. The way a handler ends sets what happens to the job:

| the handler | the job |
|---|---|
| returns normally | is done |
| returns a dict with a true `skipped` value | is done, recorded as a skip, and not retried. The acquisition methods do this when a page has no transcript or is not a PDF |
| raises `Retry` | is retried after `Retry.after` seconds or the backoff delay, and uses one attempt. Use it for work that did not fail but must run again, such as a page behind a bot wall |
| raises any other exception | is retried after the backoff delay, and dead once its attempts are spent |
| raises `Terminal` | is dead at once |

Use `Terminal` for a failure that waiting cannot fix, such as a 401, a model name that does not exist or a file that is gone. Without it, the default schedule retries a doomed job for most of a day. A job whose kind has no handler is dead at once, for the same reason.

In [ ]:
#| export
@patch
def on(self:Queue, kind:str):
    'Decorator registering the handler for `kind`.'
    def _f(fn):
        self.handlers[kind] = fn
        return fn
    return _f

@patch
def register(self:Queue, kind:str, fn):
    'Register the handler for `kind`.'
    self.handlers[kind] = fn
    return fn

@patch
def _ran(self:Queue, job, status:str, t0:float, error:str=None, result=None) -> AttrDict:
    'Write one row of run history and return what the caller reports.'
    now = time.time()
    self.runs.insert(dict(id=uuid.uuid4().hex[:12], job_id=job['id'], kind=job['kind'],
                          worker=job.get('worker'), status=status, error=(error or None),
                          started_at=t0, ended_at=now, took=round(now-t0, 3)))
    return AttrDict(job_id=job['id'], kind=job['kind'], status=status, took=round(now-t0, 3),
                    error=error, result=result)

@patch
def run_one(self:Queue, job) -> AttrDict:
    'Run the handler for one claimed job, then ack or fail the job, and record the run.'
    t0 = time.time()
    # a call from a worker that lost its lease changes nothing; the run is recorded as `lost`
    def _rep(out, status, error=None, result=None):
        if out['state'] == 'lost': status, error = 'lost', 'the lease was reclaimed mid-handler'
        return self._ran(job, status, t0, error, result)
    if (fn := self.handlers.get(job['kind'])) is None:
        err = f"no handler for {job['kind']!r}"
        return _rep(self.fail(job, err, dead=True), 'dead', err)
    try: res = fn(job['payload'])
    except Retry as e: return _rep(self.fail(job, f'Retry: {e}', after=e.after), 'retry', str(e))
    except Terminal as e:
        err = f'Terminal: {str(e)[:200]}'
        return _rep(self.fail(job, err, dead=True), 'dead', err)
    except Exception as e:
        err = f'{type(e).__name__}: {str(e)[:200]}'
        return _rep(self.fail(job, err), 'error', err)
    return _rep(self.ack(job), 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok',
                None, res)

@patch
def drain(self:Queue,
          worker:str='worker',  # lease holder recorded on each job
          limit:int=1000,       # jobs to run before returning; a ceiling, not a target
          batch:int=1,          # jobs claimed per round trip
          now:float=None
) -> L:
    "Claim and run jobs until the queue is empty or `limit` jobs have run."
    out = L()
    while len(out) < limit:
        got = self.claim(worker, n=min(batch, limit-len(out)), now=now)
        if not got: break
        out += got.map(self.run_one)
    return out

### Looking at it

`jobs` lists jobs by state and kind. `pending` counts the jobs that are waiting or running. `stats` counts the jobs in each state. `history` lists past runs.

`dead` lists the dead jobs with the error that ended each one. A failure deletes nothing. `retry` puts a dead job back on the queue and resets its attempt count. `purge` deletes finished jobs and run history older than `keep` (7 days by default). `purge` leaves dead jobs alone.

In [ ]:
#| export
@patch
def jobs(self:Queue, state:str=None, kind:str=None, limit:int=100) -> L:
    'Jobs, soonest first, filtered by state and kind.'
    w, a = [], []
    if state: w.append('state=?'); a.append(state)
    if kind:  w.append('kind=?');  a.append(kind)
    sql = 'SELECT * FROM jobs' + (' WHERE ' + ' AND '.join(w) if w else '') + ' ORDER BY run_at LIMIT ?'
    return L(self.db.q(sql, a + [limit])).map(_row)

@patch
def dead(self:Queue, limit:int=100) -> L:
    'Dead jobs: those that used up their attempts or failed with `Terminal`, each with its last error.'
    return self.jobs(state='dead', limit=limit)

@patch
def pending(self:Queue, kind:str=None) -> int:
    "Count the jobs still waiting or running, optionally of one kind."
    w, a = ('AND kind=? ', [kind]) if kind else ('', [])
    return first(self.db.q(f"SELECT count(*) n FROM jobs WHERE state IN ('ready','running') {w}", a))['n']

@patch
def retry(self:Queue, job_id:str, after:float=0) -> AttrDict:
    'Put a dead or pending job back on the queue now, with its attempt count reset.'
    now = time.time()
    self.db.q('UPDATE jobs SET state=?, attempt=0, run_at=?, error=NULL, worker=NULL, lease_until=0, '
              'updated_at=? WHERE id=?', ['ready', now+after, now, job_id])
    return first(L(self.db.q('SELECT * FROM jobs WHERE id=?', [job_id])).map(_row))

@patch
def history(self:Queue, job_id:str=None, limit:int=50) -> L:
    'Run history, newest first.'
    w, a = ('WHERE job_id=? ', [job_id]) if job_id else ('', [])
    return L(self.db.q(f'SELECT * FROM job_runs {w}ORDER BY started_at DESC LIMIT ?', a + [limit]))

@patch
def stats(self:Queue) -> dict:
    'How many jobs in each state, and when the next one is due.'
    by = {r['state']: r['n'] for r in self.db.q('SELECT state, count(*) n FROM jobs GROUP BY state')}
    nxt = first(self.db.q("SELECT min(run_at) t FROM jobs WHERE state='ready'"))
    return dict({s: by.get(s, 0) for s in STATES}, next_due=nxt['t'])

@patch
def purge(self:Queue, before:float=None) -> dict:
    'Delete finished jobs and run history older than `keep`. Dead jobs stay.'
    cut = (time.time() - self.keep) if before is None else before
    d = self.db.q("DELETE FROM jobs WHERE state='done' AND updated_at<? RETURNING id", [cut])
    r = self.db.q('DELETE FROM job_runs WHERE ended_at<? RETURNING id', [cut])
    return dict(jobs=len(d), runs=len(r))

## Ledger

Some work does not run on the queue. An app runs it in its own thread pool and shows its progress, such as a download, a model call or a build. `Ledger` keeps one row for each such job, so the app can show what runs and what ended, also after a restart. A row has `kind`, `state`, `title`, `said` (the progress line), `start`, `end`, `url`, `obj` (the thing the job is about), `why` (the reason for a failure), `seen` and `quiet`. Other keys go in the `meta` of the row.

The rows are in their own `job_ledger` table in the vault file. `Queue` does not see them: it never claims, retries or counts a ledger row. `version` goes up at each write, also in another process, so a page can ask whether anything changed.

In [ ]:
#| export
ENDED = ('done', 'failed', 'stopped', 'closed')
RUN = uuid.uuid4().hex[:12]   # this process; `close_running` closes the running rows of any other one
_LCOLS = ('key', 'kind', 'state', 'title', 'said', 'start', 'end', 'url', 'obj', 'why', 'seen', 'quiet', 'run', 'ver')

def _lrow(r) -> AttrDict|None:
    "A ledger row as an `AttrDict`: its `meta` keys merged in, and `seen` and `quiet` as bools."
    if r is None: return None
    d = {k: v for k, v in dict(r).items() if k != 'meta'}
    return AttrDict({**json.loads(r.get('meta') or '{}'), **d, 'seen': bool(d.get('seen')), 'quiet': bool(d.get('quiet'))})

class Ledger:
    "Rows for jobs that run elsewhere, such as an in-process future: their state, a progress line and a change counter."
    def __init__(self,
                 db,                      # open litesearch/fastlite Database; the rows live in its file
                 keep:float=172800,       # seconds an ended row stays, 2 days
                 run:str=None,            # the id of this process; None -> `RUN`
                 busy_ms:int=BUSY_TIMEOUT_MS):
        store_attr()
        self.run = run or RUN
        self.db.conn.set_busy_timeout(busy_ms)
        self.t = self.db.t.job_ledger
        with write_txn(self.db):
            self.t.create(key=str, kind=str, state=str, title=str, said=str, start=float, end=float, url=str, obj=str,
                          why=str, seen=int, quiet=int, run=str, ver=int, meta=str, pk='key', if_not_exists=True)
            self.db.t.job_ledger_ver.create(id=int, n=int, pk='id', if_not_exists=True)
            self.db.conn.execute('CREATE INDEX IF NOT EXISTS job_ledger_state ON job_ledger(state, "end")')

    @property
    def version(self) -> int:
        "A count that goes up at each write of a row, in any process."
        return (first(self.db.q('SELECT n FROM job_ledger_ver WHERE id=1')) or {}).get('n') or 0

    def __repr__(self):
        n = first(self.db.q("SELECT count(*) n FROM job_ledger WHERE state='running'"))['n']
        return f'Ledger({n} running, version {self.version})'

`put` merges keys into a row and makes the row when it is new. `start` writes a `running` row and `end` ends it. `get` reads one row and `rows` lists them, newest first.

In [ ]:
#| export
@patch
def _bump(self:Ledger) -> int:
    "The next version. Call inside a `write_txn`."
    return first(self.db.q('INSERT INTO job_ledger_ver(id, n) VALUES (1, 1) '
                           'ON CONFLICT(id) DO UPDATE SET n=n+1 RETURNING n'))['n']

@patch
def get(self:Ledger, key:str) -> AttrDict|None:
    "The row `key`, or None."
    return _lrow(first(self.db.q('SELECT * FROM job_ledger WHERE key=?', [key])))

@patch
def put(self:Ledger, key:str, **kv) -> AttrDict:
    "Merge `kv` into the row `key`, and make the row when it is new. A key that is not a column goes in `meta`."
    with write_txn(self.db):
        cur = first(self.db.q('SELECT * FROM job_ledger WHERE key=?', [key])) or dict(seen=0, quiet=0, run=self.run)
        meta = {**json.loads(cur.get('meta') or '{}'), **{k: v for k, v in kv.items() if k not in _LCOLS}}
        row = {**{k: v for k, v in cur.items() if k != 'meta'}, **{k: v for k, v in kv.items() if k in _LCOLS}}
        row.update(key=key, ver=self._bump(), meta=json.dumps(meta, default=str),
                   seen=int(bool(row.get('seen'))), quiet=int(bool(row.get('quiet'))))
        self.t.insert(row, replace=True)
    return _lrow(row)

@patch
def start(self:Ledger, key:str, kind:str, title:str='', said:str='', url:str='', obj:str='', quiet:bool=False, **meta) -> AttrDict:
    "Write the row `key` of a job of `kind` that runs now. Ended rows older than `keep` go."
    self.purge()
    return self.put(key, kind=kind, title=title, said=said, url=url, obj=obj, quiet=quiet, state='running',
                    start=time.time(), end=None, why='', seen=False, run=self.run, **meta)

@patch
def end(self:Ledger, key:str, state:str='done', why:str='', **kv) -> AttrDict|None:
    "End the row `key` now in `state`, with `why` for a failure. None when there is no row."
    if self.get(key) is None: return None
    return self.put(key, state=state, end=time.time(), why=why, **kv)

@patch
def rows(self:Ledger, state:str=None, kind:str=None, obj:str=None) -> L:
    "The rows, newest first, filtered by `state`, `kind` and `obj`."
    w = [(c, v) for c, v in (('state', state), ('kind', kind), ('obj', obj)) if v is not None]
    sql = 'SELECT * FROM job_ledger' + (' WHERE ' + ' AND '.join(f'{c}=?' for c, _ in w) if w else '') + ' ORDER BY start DESC'
    return L(self.db.q(sql, [v for _, v in w])).map(_lrow)

@patch
def purge(self:Ledger, before:float=None) -> int:
    "Delete the ended rows that ended before `before`, by default `keep` seconds ago. Return the count."
    cut = time.time() - self.keep if before is None else before
    with write_txn(self.db): return len(self.db.q(f'DELETE FROM job_ledger WHERE state IN ({",".join("?"*len(ENDED))}) '
                                                  'AND "end"<? RETURNING key', [*ENDED, cut]))

`track` writes the row of a future and ends it when the future ends: `done`, or `failed` with `why`. A row is never retried: the work is the caller's. A row that ended first, such as a job that you stopped, stays as it is. `say` writes the progress line of a running job. A write that fails never breaks the job.

In [ ]:
#| export
@patch
def track(self:Ledger,
          kind:str,         # what the job is
          fut,              # a `concurrent.futures.Future`, or a function that returns one, called after the row is written
          key:str=None,     # the row; None -> a new key, which the future has as `fut.key`
          ok=None,          # `ok(result) -> dict`: more keys for the row when it ends, and `state` to end in another state
          why=None,         # `why(error) -> str`: the line of a failure
          **meta            # `title`, `said`, `url`, `obj`, `quiet`, and any other keys of the row
):
    "Write a `running` row for the future `fut`, and end it when `fut` ends. Return the future."
    key = key or f'{kind}:{uuid.uuid4().hex[:12]}'
    self.start(key, kind, **meta)
    if callable(fut): fut = fut()
    try: fut.key = key
    except AttributeError: pass
    def _end(f):
        try:
            if (r := self.get(key)) is None or r.state != 'running': return
            if (e := f.exception()) is not None: self.end(key, 'failed', why=why(e) if why else f'{type(e).__name__}: {str(e)[:200]}')
            else:
                kv = dict((ok(f.result()) or {}) if ok else {})
                self.end(key, kv.pop('state', 'done'), **kv)
        except Exception: pass   # a row that cannot be written never breaks the job
    fut.add_done_callback(_end)
    return fut

@patch
def say(self:Ledger, key:str, line:str) -> bool:
    "Write the progress line `line` to the row `key` while its job runs. Whether the row changed."
    try:
        if (r := self.get(key)) is None or r.state != 'running' or r.said == line: return False
        self.put(key, said=line); return True
    except Exception: return False

A row that says `running` after a restart belongs to a job that died with its process. `close_running`, called at the start of an app, marks each such row `closed`. It closes only the rows of other processes. `seen` marks the ended rows of one object as seen, and `activity` gives the rows an app shows now: the running ones, then the ended ones not seen yet. A row that you stopped and a `quiet` row are left out.

In [ ]:
#| export
@patch
def close_running(self:Ledger) -> int:
    "Mark `closed` each running row that another process wrote. Return the count."
    ks = self.db.q("SELECT key FROM job_ledger WHERE state='running' AND (run IS NULL OR run!=?)", [self.run])
    for r in ks: self.end(r['key'], 'closed')
    self.purge()
    return len(ks)

@patch
def seen(self:Ledger, obj:str) -> int:
    "Mark each ended row of the object `obj` as seen. Return the count that changed."
    ks = [r.key for r in self.rows(obj=obj) if r.state in ENDED and not r.seen]
    for k in ks: self.put(k, seen=True)
    return len(ks)

@patch
def activity(self:Ledger,
             fresh:float=21600,   # seconds an ended row shows until it is seen, 6 hours
             now:float=None) -> L:
    "The running rows, newest first, then the rows that ended in the last `fresh` seconds and are not seen, newest first."
    now, rs = now or time.time(), self.rows().filter(lambda r: not r.quiet)
    run = rs.filter(lambda r: r.state == 'running')
    new = sorted((r for r in rs if r.state in ('done', 'failed', 'closed') and not r.seen and now - (r.end or 0) < fresh),
                 key=lambda r: -(r.end or 0))
    return run + L(new)

## Try it

In [ ]:
from litesearch import database

db = database()
q  = Queue(db, lease=60, max_attempts=3, base=1)
seen = []
q.register('echo', lambda p: seen.append(p['n']) or dict(n=p['n']))
q.enqueue_all('echo', [dict(n=i) for i in range(3)])
q.drain('w1')
seen, q.stats()

In [ ]:
#| hide
# claim is atomic: two workers over one ready job get it once between them, never twice
db2 = database()
q2  = Queue(db2, lease=60, max_attempts=3, base=1)
q2.enqueue('t', dict(a=1))
a, b = q2.claim('w1', n=5), q2.claim('w2', n=5)
test_eq(len(a) + len(b), 1)
test_eq(first(a or b)['payload'], dict(a=1))
test_eq(first(a or b)['state'], 'running')
# ...and a claimed job is no longer due, so a third worker sees nothing
test_eq(q2.claim('w3', n=5), [])

In [ ]:
#| hide
# a failure backs off and comes back; the third one is dead, and the payload survives to be read
q3 = Queue(database(), lease=60, max_attempts=3, base=100)
j  = q3.enqueue('t', dict(url='http://x'))
for i in (1, 2):
    c = first(q3.claim('w'))
    out = q3.fail(c, f'boom {i}')
    test_eq(out['state'], 'ready')
    test_eq(out['attempt'], i)
    assert out['run_at'] > time.time(), 'a retry is scheduled in the future'
    q3.db.q("UPDATE jobs SET run_at=0 WHERE id=?", [j['id']])   # skip the wait
c = first(q3.claim('w'))
test_eq(q3.fail(c, 'boom 3')['state'], 'dead')
test_eq(len(q3.dead()), 1)
test_eq(q3.dead()[0]['payload'], dict(url='http://x'))
test_eq(q3.dead()[0]['error'], 'boom 3')
test_eq(q3.claim('w'), [])                     # dead jobs are not due
test_eq(q3.retry(j['id'])['state'], 'ready')   # ...until a human puts one back
test_eq(len(first(q3.claim('w')) and [1]), 1)

In [ ]:
#| hide
# a worker killed mid-job never acks; the lease expires and the job comes back
q4 = Queue(database(), lease=30, max_attempts=3, base=1)
q4.enqueue('t', dict(a=1))
t = time.time()
test_eq(len(q4.claim('doomed', now=t)), 1)
test_eq(q4.reclaim(now=t+10), [])              # lease still held
back = q4.reclaim(now=t+31)
test_eq(len(back), 1)
test_eq(back[0]['state'], 'ready')
test_eq(back[0]['attempt'], 1)                 # a job that kills its worker every time still dies
test_eq(back[0]['error'], 'lease expired')

In [ ]:
#| hide
# a handler slower than its lease loses the job underneath it, and its late ack is refused
q9 = Queue(database(), lease=30, max_attempts=5, base=1)
q9.enqueue('t', dict(a=1))
t = time.time()
slow = first(q9.claim('slow', now=t))
test_eq(len(q9.reclaim(now=t+31)), 1)                     # the poller takes it back
fast = first(q9.claim('fast', now=t+3600))
test_eq(fast['id'], slow['id'])
test_eq(q9.ack(slow)['state'], 'lost')                    # slow finishes late and is refused
test_eq(first(q9.jobs())['state'], 'running')             # ...so fast still holds it
test_eq(q9.fail(slow, 'late')['state'], 'lost')           # and a late failure cannot resurrect it
test_eq(q9.ack(fast)['state'], 'done')
test_eq(q9.stats(), dict(ready=0, running=0, done=1, dead=0, next_due=None))
# a failing handler that lost its lease is reported the same way, and does not charge an attempt
q9.enqueue('t')
t = time.time()
slow = first(q9.claim('slow', now=t))
q9.reclaim(now=t+31)
test_eq(q9.fail(slow, 'boom')['state'], 'lost')
test_eq(first(q9.jobs(state='ready'))['attempt'], 1)      # the reclaim's attempt, not a second one

# the same through run_one: the handler ran, the ack was refused, and the history records it
q10 = Queue(database(), lease=30, max_attempts=5, base=1)
q10.enqueue('t')
q10.register('t', lambda p: q10.reclaim(now=time.time()+31) and dict(ok=1))   # lease lost mid-handler
r = first(q10.drain('slow'))
test_eq(r['status'], 'lost')
test_eq(r['result'], dict(ok=1))                          # the work happened; only the ack did not
test_eq(first(q10.jobs())['state'], 'ready')              # reclaim left it due again, for anyone

# two pollers over one expired lease reclaim it once between them, not twice
q11 = Queue(database(), lease=30, max_attempts=5, base=1)
q11.enqueue('t')
t = time.time()
q11.claim('doomed', now=t)
test_eq(len(q11.reclaim(now=t+31)), 1)
test_eq(len(q11.reclaim(now=t+31)), 0)                    # nothing left to take back
test_eq(first(q11.jobs())['attempt'], 1)                  # one kill is one attempt

In [ ]:
#| hide
# `key` is what makes a re-run of a fan-out cheap: the same key pending twice is one job
q5 = Queue(database(), base=1)
a = q5.enqueue('file', dict(p='/x'), key='/x')
b = q5.enqueue('file', dict(p='/x'), key='/x')
test_eq(a['id'], b['id'])
test_eq(len(q5.jobs()), 1)
q5.register('file', lambda p: dict(ok=1))
q5.drain('w')
c = q5.enqueue('file', dict(p='/x'), key='/x')  # done, so the same work can be asked for again
test_ne(c['id'], a['id'])
test_eq(len(q5.jobs()), 2)

In [ ]:
#| hide
# the three handler outcomes: ok, a skip that is final, and a Retry that is not
q6 = Queue(database(), max_attempts=3, base=1)
q6.register('ok',   lambda p: dict(added=1))
q6.register('skip', lambda p: dict(skipped='no transcript'))
q6.register('wall', lambda p: (_ for _ in ()).throw(Retry('bot wall', after=5)))
q6.register('bad',  lambda p: 1/0)
q6.register('401',  lambda p: (_ for _ in ()).throw(Terminal('no such model')))
for k in ('ok', 'skip', 'wall', 'bad', '401'): q6.enqueue(k)
r = {x['kind']: x for x in q6.drain('w')}
test_eq(r['ok']['status'], 'ok')
test_eq(r['skip']['status'], 'skipped')
test_eq(r['wall']['status'], 'retry')
test_eq(r['bad']['status'], 'error')
test_eq(r['401']['status'], 'dead')
assert 'ZeroDivisionError' in r['bad']['error']
test_eq({j['kind'] for j in q6.jobs(state='done')}, {'ok', 'skip'})   # a skip is not a failure
test_eq({j['kind'] for j in q6.jobs(state='ready')}, {'wall', 'bad'})
# Terminal spends no attempts: 1 of 3, dead anyway
test_eq([(j['attempt'], j['state']) for j in q6.jobs(kind='401')], [(1, 'dead')])
test_eq(len(q6.history()), 5)
# a kind nothing subscribes to is dead on the first attempt, not after three
q6.enqueue('nobody')
test_eq(first(q6.drain('w', limit=1))['status'], 'dead')
test_eq([j['state'] for j in q6.jobs(kind='nobody')], ['dead'])

In [ ]:
#| hide
# priority orders the queue, and a job scheduled ahead is not claimed early
q7 = Queue(database(), base=1)
t = time.time()
q7.enqueue('t', dict(n='later'),  priority=0,  run_at=t+3600)
q7.enqueue('t', dict(n='low'),    priority=5,  run_at=t)
q7.enqueue('t', dict(n='urgent'), priority=-1, run_at=t)
test_eq([j['payload']['n'] for j in q7.claim('w', n=9, now=t)], ['urgent', 'low'])
test_eq([j['payload']['n'] for j in q7.claim('w', n=9, now=t+3601)], ['later'])

In [ ]:
#| hide
# purge clears finished work and its history, and leaves the dead letter alone
q8 = Queue(database(), max_attempts=1, base=1)
q8.register('ok', lambda p: dict(ok=1))
q8.enqueue('ok'); q8.enqueue('nope')
q8.drain('w')
test_eq(q8.stats()['done'], 1)
test_eq(q8.stats()['dead'], 1)
test_eq(q8.purge(before=time.time()+1), dict(jobs=1, runs=2))
test_eq(q8.stats(), dict(ready=0, running=0, done=0, dead=1, next_due=None))

In [ ]:
#| hide
from concurrent.futures import ThreadPoolExecutor, Future
from litesearch import database
lg = Ledger(database())
with ThreadPoolExecutor(2) as ex:
    f1 = lg.track('build', ex.submit(lambda: 42), key='b1', title='Build', obj='walk:1', ok=lambda r: dict(n=r))
    f2 = lg.track('build', lambda: ex.submit(lambda: 1/0), title='Fails', obj='walk:1', why=lambda e: 'It did not work.')
test_eq((lg.get('b1').state, lg.get('b1').n, lg.get('b1').title), ('done', 42, 'Build'))   # ok() keys go in meta
test_eq((lg.get(f2.key).state, lg.get(f2.key).why), ('failed', 'It did not work.'))
# no retry: nothing is queued, and the queue's tables are not touched
test_eq('jobs' in lg.db.t, False)
# say writes the line of a running job only, and only when it changed
lg.start('d1', 'download', title='Model', said='0%')
test_eq((lg.say('d1', '50%'), lg.say('d1', '50%'), lg.get('d1').said), (True, False, '50%'))
test_eq(lg.say('b1', 'late'), False)
# a row that ended first stays: a stopped job does not become done
f3 = Future(); lg.track('sort', f3, key='s1'); lg.end('s1', 'stopped'); f3.set_result(None)
test_eq(lg.get('s1').state, 'stopped')
# version only goes up, purge included
v0 = lg.version; lg.put('d1', said='60%'); v1 = lg.version
lg.purge(before=time.time() + 1); test_eq((v1 > v0, lg.version >= v1), (True, True))
test_eq(lg.get('b1'), None)                                                   # the ended rows went
test_eq(lg.get('d1').state, 'running')                                        # a running row stays

In [ ]:
#| hide
# the rows survive a restart, and close_running closes only the rows of the process before
from tempfile import mkdtemp
from pathlib import Path
_p = str(Path(mkdtemp())/'vault.db')
old = Ledger(database(_p), run='before')
old.start('w1', 'walk', title='Walk', obj='walk:1'); old.start('w2', 'walk', quiet=True)
old.start('w3', 'walk', title='Done walk', obj='walk:1'); old.end('w3')
new = Ledger(database(_p), run='after')
new.start('w4', 'walk', title='New walk')
test_eq(sorted(r.key for r in new.rows()), ['w1', 'w2', 'w3', 'w4'])
test_eq(new.close_running(), 2)
test_eq([new.get(k).state for k in ('w1', 'w2', 'w3', 'w4')], ['closed', 'closed', 'done', 'running'])
# activity: running first, then the ended rows not seen; a quiet row stays out
test_eq(new.activity().attrgot('key'), ['w4', 'w1', 'w3'])
test_eq(new.seen('walk:1'), 2)
test_eq(new.activity().attrgot('key'), ['w4'])
test_eq(Ledger(database(_p)).version, new.version)                            # one counter in the file

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()